## Import

In [56]:
import psycopg2
from psycopg2.extras import execute_values
import random
import datetime
from dateutil.relativedelta import relativedelta

In [57]:
conn = psycopg2.connect(
    host="aws-1-ap-south-1.pooler.supabase.com",
    database="postgres",
    user="postgres.rruavcjmtgpxyznzwkhw",
    password="khaibaolocnguyen",
    port=5432
)
cur = conn.cursor()

## Lấy ra danh sách Invoice đã tạo trước đó

In [68]:
conn.rollback()
query = """
  SELECT invoice.invoice_id, invoice.created_at, invoice.customer_id
  FROM invoice
 """
cur.execute(query)
invoices = cur.fetchall()

def format_invoice(row):
    return {
        "id": row[0],
        "created_at": row[1],
        "customer_id": row[2]
    }

invoices = [format_invoice(row) for row in invoices]
print(len(invoices))
for invoice in invoices:
    print(invoice)

1412
{'id': '5dfe7679-9c00-49b9-9d74-6602ccfc725a', 'created_at': datetime.datetime(2025, 8, 23, 4, 19, 6), 'customer_id': '1e16fb23-2e70-47a1-b0f6-143184013d3e'}
{'id': '09da38be-b2c5-46b1-87a3-0e5dad0f1e8e', 'created_at': datetime.datetime(2025, 1, 18, 17, 3, 45), 'customer_id': '54158d66-865b-4fa3-990f-ad00dd196916'}
{'id': 'b80abb42-3520-4013-bd2f-100b8f160e9a', 'created_at': datetime.datetime(2025, 4, 24, 1, 16, 1), 'customer_id': '0f4a66a1-bd0f-4f05-b273-feb857529798'}
{'id': '1a5c457a-d227-452b-9bb1-3a3599e04c0f', 'created_at': datetime.datetime(2025, 3, 4, 3, 43, 55), 'customer_id': 'dcfb649b-43e1-4cf0-a4dd-74ddb928248b'}
{'id': 'b2c6018e-1455-487d-a5ea-c51a5f2a698a', 'created_at': datetime.datetime(2025, 5, 14, 23, 57, 12), 'customer_id': '79de3a80-e558-4705-9fad-b92128743187'}
{'id': '635ca150-d177-456c-88d8-d7d8ff5fd891', 'created_at': datetime.datetime(2025, 7, 21, 3, 4, 40), 'customer_id': '5e1d21ab-55c2-49d8-81c7-8e9ed785827d'}
{'id': '6f8b7c0d-3591-43b7-9678-c6aeac288273

## Lấy danh sách service và branch available của nó, mỗi branch lấy ra danh sách bác sĩ làm việc ở đó

In [59]:
conn.rollback()

query = """
  SELECT service.service_id, service.service_name, branchservice.branch_id
  FROM service
  JOIN branchservice ON service.service_id = branchservice.service_id
"""

cur.execute(query)
rows = cur.fetchall()

services_map = {}

for service_id, service_name, branch_id in rows:
    if service_id not in services_map:
        services_map[service_id] = {
            "id": service_id,
            "name": service_name,
            "branch_id": []
        }
    services_map[service_id]["branch_id"].append(branch_id)

services = list(services_map.values())


# Với mỗi branch trong từng service, lấy ra danh sách bác sĩ của branch đó
for service in services:
    for branch_id in service["branch_id"]:
        query = """
          SELECT employee_id
          FROM employeehistory
          WHERE branch_id = %s
        """
        cur.execute(query, (branch_id,))
        rows = cur.fetchall()
        veterinarians = [row[0] for row in rows]
        service["veterinarians"] = veterinarians

print(len(services))
for service in services:
    print(service["id"])
    print(service["name"])
    print(service["branch_id"])
    print(service["veterinarians"])

3
53e5a498-1eae-43f3-9a76-90f1fbebe535
Medical Examination
['0e00def4-2be9-405f-bb3e-1a2bc888828b', 'a49d5373-fdae-4e7e-b90f-17344d0a6aaf', '6d1f53af-555c-4296-9066-e304432d0633', 'c26300f2-703c-4022-82c0-b68dc260aa4f', '6feeeb11-f446-46da-9214-cd004151c492', '8adb1159-cee3-4c59-b08f-3a9aa369d73e', '29e07c3a-47bf-4fef-8fbe-471e1587118c', 'ad3cc27c-6b13-46ed-bcfd-cc099f0d5d47', '7192a064-1e01-42c7-87b9-6e6c3d381339', '015cf519-76c9-478e-9bf8-e25bb91f92bb']
['65822fa7-1c92-4ff6-a838-a36f65020bd4', 'c9a54ed4-b78b-4a21-bfed-9d81209d5e99', '49da80ed-55ff-4246-8c08-36d09c595677', '49da80ed-55ff-4246-8c08-36d09c595677', 'cf7b7682-b90f-49a3-a2f2-2f91b2ecde83', '635d6c07-d42c-4333-aa6a-5ed35ab3324c', '142d238f-1fe9-4e8d-b97e-929921d5876e', '21d8f6f1-6142-46bf-a54a-778e6d436f58', '82204f84-37ae-4cf8-8e6d-e19428f1c561', '3422e5be-5054-482a-a695-9c1e5bbe4427', 'dfb74b07-05f3-45ed-9e95-d42e17e0de8a', 'e3de82cc-a9c3-4bd6-83c3-e4b46a85f9b2', '588d470d-84f8-4de3-901d-96e2496da54b', '828a246e-6ba3-4ef5

## Lấy ra danh sách khách hàng và thú cưng của họ

In [60]:
query = """
  SELECT customer.customer_id, pet.pet_id
  FROM customer
  JOIN pet ON customer.customer_id = pet.customer_id
"""

cur.execute(query)
rows = cur.fetchall()

customers_map = {}

for customer_id, pet_id in rows:
    if customer_id not in customers_map:
        customers_map[customer_id] = {
            "id": customer_id,
            "pets": []
        }
    customers_map[customer_id]["pets"].append(pet_id)

customers = list(customers_map.values())

print(len(customers))
for customer in customers:
    print(customer["id"])
    print(customer["pets"])

704
41ac49fe-c947-4b2c-9bb3-c37e33788ab8
['f044b82a-d064-40ed-9843-87a7f11a75b6', '24d68156-300d-4882-ab92-1b811e41d06d', 'f11790eb-3102-47c5-9045-c906852a8970']
732543aa-71f2-4a71-8acc-ad1475bc5419
['32abf3c5-d949-4127-8e10-498e15529a9f', '130b07ed-76a5-4e5e-90b7-4398a8cc7a15', '4af9df9f-59a0-49a6-8109-579a07892182', '8cb2973d-6ee6-44f2-a594-ff0abc619f8c', '086f8262-1fe0-4c70-84c8-e9adba42ccd2', 'a15be1d5-1b2c-4af3-9a4b-6972f586ab4a', '7a611551-e27f-44db-92b2-5083aaff98e0', 'c59b5957-3da3-44ab-b2e3-9696fe0e5901', 'ace96c17-1350-454f-825a-f7e7ca458347', '7b0e5009-42c8-4626-b861-5a443626d8a4']
c838dc90-258f-4a7f-9604-4e0765a4299b
['3797cd64-98c6-480c-9294-2b4e58fb8ecd']
0fc8eda2-367a-4f40-baae-e1e3fc9df975
['0fcf9f03-eab6-4d89-bfc2-f728c1c030b9', 'd818322c-ebbd-4aea-b40e-97c661d119d0', '501051f3-676f-4840-864b-6cf3cb5a93a3']
e8feb2fc-a47d-4073-9d2d-bf6aebb17e1a
['e9ef86ab-2d1f-4f5a-b3f8-e3c0552f5e1d', '5dc0b537-3983-47b6-9c82-c4c0437c576e', '0ed2e427-6e6f-4332-87da-87fe07f39c53']
a7400f

## Lấy ra danh sách Vaccine

In [61]:
conn.rollback()
query = """
  SELECT vaccine.vaccine_id
  FROM vaccine
 """
cur.execute(query)
vaccines = cur.fetchall()

vaccines = [row[0] for row in vaccines]
print(len(vaccines))
for vaccine in vaccines:
    print(vaccine)

10029
9ce6082c-d898-4190-9f86-aefd45c45702
0dff9336-23d0-45c4-839e-e5f7b3c1851f
a9717161-2476-432e-8f3f-f09c483db8c1
8bf02720-ba6a-43d0-a3ac-61fa61ac7b61
3441309e-ce39-4dcd-8a33-4c3edae8d20d
4f3e2a72-46df-44ec-97bb-2a3786960096
17e8e89d-f4d8-4eb3-89fe-4191a7fb81c1
edb5893d-7085-44e6-8861-7f05ea107ee5
053865bc-34e8-4432-829e-e525fa4db558
40fdb509-9d18-4b13-846d-a3d5a2a12014
f872fbbb-a81e-4879-b08c-a33f3a8406c6
73039224-4b3b-47dd-b77d-b91f247395fe
b0470b50-45cb-447e-943f-597c7f476cec
c729a9b9-e909-49fa-b33a-ad3a42d66866
25eca66f-49c5-4247-838e-aa75681db22c
c689ef9f-6dde-4ba1-bd0d-9fc429c998e0
649f5668-6d21-4289-aae3-3bdeda678b3d
4cf4327e-7dfc-4abd-9680-fd79ebe0ce27
b7838753-b5e5-4f36-ac0d-0856f9948068
2af831d9-3340-4978-9fbc-3722934dc875
4d9328d3-f612-40b4-9426-9385a47e4a79
ca699368-d5ef-4aba-bb8e-f3b54d368d23
409d5d6c-67a8-4233-8b4b-93c7a3dce51c
6487c0e3-45c2-4d31-b287-ff2aed63c6c1
258f4c5c-ea2b-4574-802c-acac7defd04f
95ac2206-06bf-4328-bd39-683d349fd332
50c7fa6e-94eb-4972-9055-cfb8c3df

## Lấy ra danh sách Vaccine Package

In [62]:
conn.rollback()

query = """
  SELECT package_id, duration
  FROM vaccinationpackage
"""
cur.execute(query)
rows = cur.fetchall()

# format thành list dict
packages = [{"id": row[0], "duration": row[1]} for row in rows]

print(len(packages))
for package in packages:
    print(package)

1007
{'id': '2a3e3170-6ab2-447d-a1ac-c27a355cc948', 'duration': Decimal('6')}
{'id': 'dea4ffd4-b30f-4892-928b-df143dfa1242', 'duration': Decimal('3')}
{'id': 'd9492f95-7ad0-4fb8-8a76-bdd93df49b0a', 'duration': Decimal('6')}
{'id': '002028f6-27d7-4102-a25e-0a2b1dc9f988', 'duration': Decimal('90')}
{'id': '5b533719-8da9-4a08-87f2-0dcfe179581a', 'duration': Decimal('12')}
{'id': 'f8e9164c-462a-4eb0-ae33-efe6da6ffc14', 'duration': Decimal('6')}
{'id': '091efdf9-cc2c-4bd2-9cb9-45f8c700c667', 'duration': Decimal('180')}
{'id': 'b583d058-80bf-4175-81d8-8023e45d930b', 'duration': Decimal('90')}
{'id': '23b1f3bc-26ad-428d-aa74-ee2e45e0259e', 'duration': Decimal('1')}
{'id': '2e6d0c0f-d284-43d4-a2aa-82b9d765d721', 'duration': Decimal('5')}
{'id': '4d5de644-de4c-41be-99aa-6297dfe8908d', 'duration': Decimal('180')}
{'id': '450838cc-b05f-446a-b179-e6867f17fca2', 'duration': Decimal('90')}
{'id': 'ad7c2522-5897-4254-b900-63268ebfb7a0', 'duration': Decimal('90')}
{'id': 'c91ed2d3-c75e-4840-a5e3-f9a9c

## Main

In [ ]:
def random_datetime(start, end):
    delta = end - start
    return start + datetime.timedelta(
        seconds=random.randint(0, int(delta.total_seconds()))
    )

start_dt = datetime.datetime(2025, 1, 1)
end_dt = datetime.datetime.now()

# Mảng symptom
symptoms = [
    "Fever", "Cough", "Vomiting", "Diarrhea", "Lethargy",
    "Loss of appetite", "Itching", "Sneezing", "Limping", "Swelling"
]

# Mảng diagnosis
diagnosis = [
    "Parvovirus", "Feline Leukemia", "Dermatitis", "Ear infection",
    "Gastroenteritis", "Arthritis", "Allergy", "Respiratory infection"
]

# Mảng prescription
prescriptions = [
    "Amoxicillin", "Prednisone", "Metronidazole", "Carprofen",
    "Fipronil", "Vitamin supplements", "Probiotics", "Antihistamine"
]

for invoice in invoices:
    num_item = random.randint(1, 4)
    for idx in range(num_item):
        service_types = ["Medical Examination", "Vaccine Single Service", "Vaccine Package Service"]
        selected_type = random.choice(service_types)
        services_of_type = [s for s in services if s["name"] == selected_type]
        service_selected = random.choice(services_of_type)
        target_customer = next(
            (c for c in customers if c["id"] == invoice["customer_id"]),
            None
        )
        # Tạo mới servicebooking
        new_servicebooking = {
            "date": random_datetime(start_dt, end_dt),
            "status": "completed",
            "price": random.randint(500000, 10000000),
            "service_id": service_selected["id"],
            "branch_id": random.choice(service_selected["branch_id"]),
            "employee_id": random.choice(service_selected["veterinarians"]),
            "pet_id": random.choice(target_customer["pets"]),
            "invoice_id": invoice["id"]
        }
        query = """
          INSERT INTO servicebooking (date, status, price, service_id, branch_id, employee_id, pet_id, invoice_id)
          VALUES (%s, %s, %s, %s, %s, %s, %s, %s)
          RETURNING booking_id;
        """
        cur.execute(query, (
            new_servicebooking["date"],
            new_servicebooking["status"],
            new_servicebooking["price"],
            new_servicebooking["service_id"],
            new_servicebooking["branch_id"],
            new_servicebooking["employee_id"],
            new_servicebooking["pet_id"],
            new_servicebooking["invoice_id"]
        ))
        booking_id = cur.fetchone()[0]

        # Nếu là Medical Examination
        if service_selected["name"] == "Medical Examination":
            new_medicalexamination = {
                "booking_id": booking_id,
                "symptom": ", ".join(random.sample(symptoms, random.randint(1, 3))),
                "diagnosis": ", ".join(random.sample(diagnosis, random.randint(1, 2))),
                "prescription": ", ".join(random.sample(prescriptions, random.randint(1, 2))),
                "next_appointment": random_datetime(new_servicebooking["date"], end_dt)
            }
            query = """
              INSERT INTO medicalexamination (booking_id, symptom, diagnosis, prescription, next_appointment)
              VALUES (%s, %s, %s, %s, %s);
            """
            cur.execute(query, (
                new_medicalexamination["booking_id"],
                new_medicalexamination["symptom"],
                new_medicalexamination["diagnosis"],
                new_medicalexamination["prescription"],
                new_medicalexamination["next_appointment"]
            ))

        # Nếu là Vaccine Single Service
        if service_selected["name"] == "Vaccine Single Service":
            new_vaccine_single = {
                "booking_id": booking_id,
                "vaccine_id": random.choice(vaccines),
                "dosage": f"{random.randint(5, 15)} ml"
            }
            query = """
              INSERT INTO vaccinationsingleservice (booking_id, vaccine_id, dosage)
              VALUES (%s, %s, %s);
            """
            cur.execute(query, (
                new_vaccine_single["booking_id"],
                new_vaccine_single["vaccine_id"],
                new_vaccine_single["dosage"]
            ))

        # Nếu là Vaccine Package Service
        if service_selected["name"] == "Vaccine Package Service":
            package_selected = random.choice(packages)
            start_date = random_datetime(start_dt, end_dt)
            end_date = start_date + relativedelta(months=package_selected["duration"])
            new_vaccine_package = {
                "booking_id": booking_id,
                "package_id": package_selected["id"],
                "start_date": start_date,
                "end_date": end_date
            }
            query = """
              INSERT INTO vaccinationpackageservice (booking_id, package_id, start_date, end_date)
              VALUES (%s, %s, %s, %s);
            """
            cur.execute(query, (
                new_vaccine_package["booking_id"],
                new_vaccine_package["package_id"],
                new_vaccine_package["start_date"],
                new_vaccine_package["end_date"]
            ))
    conn.commit()

## (Optional) Update giá

In [73]:
conn.rollback()
# Lấy tất cả booking_id hiện có
cur.execute("SELECT booking_id FROM servicebooking")
bookings = [row[0] for row in cur.fetchall()]

# Tạo danh sách giá random tương ứng với từng booking
update_data = [(random.randint(500000, 10000000), str(booking_id)) for booking_id in bookings]

# Update batch tất cả servicebooking cùng lúc với explicit cast sang UUID
query_update = """
UPDATE servicebooking AS sb
SET price = data.price
FROM (VALUES %s) AS data(price, booking_id)
WHERE sb.booking_id = data.booking_id::uuid
"""
execute_values(cur, query_update, update_data)

conn.commit()